In [22]:
# ============================================================
# NOTEBOOK 1 - GEE Forest Area Computation
# University of Pittsburgh | Biochar Feedstock Methodology
# ============================================================

In [74]:
# ── CELL 1: Installation & Authentication ────────────────────────────────────
!pip install -q earthengine-api geemap

import ee
import geemap
import os
os.environ['GOOGLE_MAPS_API_KEY'] = ''
import pandas as pd

from google.colab import drive
drive.mount ('/content/drive')

PROJECT_FOLDER = '/content/drive/MyDrive/BiocharProject/'

ee.Authenticate()
ee.Initialize(project = 'spry-blade-487218-n0')

print('✅ Libraries imported')
print('✅ Google Drive mounted')
print('✅ GEE connected')
print(f'✅ Project folder: {PROJECT_FOLDER}')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Libraries imported
✅ Google Drive mounted
✅ GEE connected
✅ Project folder: /content/drive/MyDrive/BiocharProject/


In [24]:
# ── CELL 2: Load Datasets ─────────────────────────────────────────────────────
Hansen_GFC_2024      = ee.Image("UMD/hansen/global_forest_change_2024_v1_12")
GLC_FSC30D_annual    = ee.ImageCollection('projects/sat-io/open-datasets/GLC-FCS30D/annual')
GLC_FSC30D_five_year = ee.ImageCollection('projects/sat-io/open-datasets/GLC-FCS30D/five-years-map')

print('✅ Hansen GFC 2024 loaded')
print('✅ GLC FCS30D annual loaded')
print('✅ GLC FCS30D five-year loaded')

✅ Hansen GFC 2024 loaded
✅ GLC FCS30D annual loaded
✅ GLC FCS30D five-year loaded


In [25]:
#── CELL 3: Select & Mask Hansen Bands
treecover2000 = Hansen_GFC_2024.select('treecover2000')
datamask = Hansen_GFC_2024.select('datamask')

#mask: keep pixels with tree cover > 0 AND valid land pixels 
treecover2000_masked = treecover2000.updateMask(treecover2000.gt(0)).updateMask(datamask.eq(1))

print('✅ treecover2000 masked')


✅ treecover2000 masked


In [26]:
# ── CELL 4: Load GLC FSC30D year 2000
glc_2000 = GLC_FSC30D_annual.mosaic().select('b1')

#forest mask: classes 51 to 92 are forest classes
glc_2000_forest = glc_2000.gte(51).And(glc_2000.lte(92)).selfMask()
print ('✅ GLC FSC30D 2000 loaded')
print ('✅ GLC FSC30D 2000 forest mask loaded')


# ── Forest Class Definitions ─────────────────────────────────────────

forestClasses = [
    {'code': 51, 'color': '4c7300', 'name': '51 Open evergreen broadleaved'},
    {'code': 52, 'color': '006400', 'name': '52 Closed evergreen broadleaved'},
    {'code': 61, 'color': 'a8c800', 'name': '61 Open deciduous broadleaved'},
    {'code': 62, 'color': '00a000', 'name': '62 Closed deciduous broadleaved'},
    {'code': 71, 'color': '005000', 'name': '71 Open evergreen needleleaved'},
    {'code': 72, 'color': '003c00', 'name': '72 Closed evergreen needleleaved'},
    {'code': 81, 'color': '286400', 'name': '81 Open deciduous needleleaved'},
    {'code': 82, 'color': '285000', 'name': '82 Closed deciduous needleleaved'},
    {'code': 91, 'color': 'a0b432', 'name': '91 Open mixed forest'},
    {'code': 92, 'color': '788200', 'name': '92 Closed mixed forest'},
]

print(f'{len(forestClasses)} forest classes defined')

✅ GLC FSC30D 2000 loaded
✅ GLC FSC30D 2000 forest mask loaded
10 forest classes defined


In [27]:
# ── CELL 5: FAO Regional Classification (LSIB-aligned) ───────────────────────

FAO_LSIB_REGION = {
    "Africa": {
        "Central Africa": [
            "Cameroon", "Central African Rep", "Dem Rep of the Congo",
            "Rep of the Congo", "Equatorial Guinea", "Gabon", "Sao Tome & Principe"
        ],
        "East Africa": [
            "Burundi", "Djibouti", "Eritrea", "Ethiopia", "Kenya", "Rwanda",
            "Somalia", "Sudan", "Uganda"
        ],
        "Indian Ocean Islands": [
            "Comoros", "Madagascar", "Mauritius", "Seychelles"
        ],
        "Southern Africa": [
            "Angola", "Botswana", "Lesotho", "Malawi", "Mozambique", "Namibia",
            "South Africa", "Swaziland", "Tanzania", "Zambia", "Zimbabwe"
        ],
        "West Africa": [
            "Benin", "Burkina Faso", "Cabo Verde", "Chad", "Cote d'Ivoire",
            "Gambia, The", "Ghana", "Guinea", "Guinea-Bissau", "Liberia",
            "Mali", "Mauritania", "Niger", "Nigeria", "Senegal", "Sierra Leone", "Togo"
        ],
    },
    "Americas": {
        "Caribbean": [
            "Antigua & Barbuda", "Bahamas, The", "Barbados", "Belize", "Cuba",
            "Dominica", "Dominican Republic", "Grenada", "Guyana", "Haiti",
            "Jamaica", "St Kitts & Nevis", "Saint Lucia",
            "St Vincent & the Grenadines", "Suriname", "Trinidad & Tobago"
        ],
        "Central America and Mexico": [
            "Costa Rica", "El Salvador", "Guatemala", "Honduras",
            "Mexico", "Nicaragua", "Panama"
        ],
        "North America": ["Canada", "United States"],
        "South America": [
            "Argentina", "Bolivia", "Brazil", "Chile", "Colombia",
            "Ecuador", "Paraguay", "Peru", "Uruguay", "Venezuela"
        ],
    },
    "Europe": {
        "Eastern Europe": [
            "Albania", "Armenia", "Belarus", "Bosnia & Herzegovina", "Bulgaria",
            "Croatia", "Czechia", "Estonia", "Georgia", "Hungary", "Latvia",
            "Lithuania", "Montenegro", "Poland", "Moldova", "Romania",
            "Russia", "Serbia", "Slovakia", "Slovenia", "Macedonia", "Ukraine"
        ],
        "Western Europe": [
            "Andorra", "Austria", "Belgium", "Denmark", "Finland", "France",
            "Germany", "Greece", "Iceland", "Ireland", "Italy", "Liechtenstein",
            "Luxembourg", "Monaco", "Netherlands", "Norway", "Portugal",
            "San Marino", "Spain", "Sweden", "Switzerland", "United Kingdom"
        ],
    },
    "Near East": {
        "Central Asia": [
            "Azerbaijan", "Kazakhstan", "Kyrgyzstan", "Tajikistan",
            "Turkmenistan", "Uzbekistan"
        ],
        "South/East Mediterranean": [
            "Algeria", "Cyprus", "Egypt", "Israel", "Jordan", "Lebanon",
            "Libya", "Malta", "Morocco", "Syria", "Tunisia", "West Bank"
        ],
        "West Asia": [
            "Afghanistan", "Bahrain", "Iran", "Iraq", "Kuwait", "Oman",
            "Pakistan", "Qatar", "Saudi Arabia", "Turkey",
            "United Arab Emirates", "Yemen"
        ],
    },
    "Asia and the Pacific": {
        "East Asia": ["China", "Korea, North", "Japan", "Mongolia", "Korea, South"],
        "Pacific Region": [
            "Australia", "Cook Is", "Fiji", "Kiribati", "Marshall Is",
            "Fed States of Micronesia", "Nauru", "New Zealand", "Niue",
            "Palau", "Papua New Guinea", "Samoa", "Solomon Is",
            "Tonga", "Tuvalu", "Vanuatu"
        ],
        "South Asia": ["Bangladesh", "Bhutan", "India", "Maldives", "Nepal", "Sri Lanka"],
        "Southeast Asia": [
            "Brunei", "Cambodia", "Indonesia", "Laos", "Malaysia", "Burma",
            "Philippines", "Singapore", "Thailand", "Timor-Leste", "Vietnam"
        ],
    },
}

print(f'✅ FAO_LSIB_REGION defined - {len(FAO_LSIB_REGION)} regions')

✅ FAO_LSIB_REGION defined - 5 regions


In [28]:
# ── CELL 6: Helper Functions ──────────────────────────────────────────────────
def get_all_countries(regions):
    """Return a flat sorted list of all country names."""
    countries = []
    for region in regions.values():
        for subregion_countries in region.values():
            countries.extend(subregion_countries)
    return sorted (countries)

def build_country_lookup(regions):
    """Return a dict mapping country name --> {region, subregion}"""
    lookup = {}
    for region_name, subregions in regions.items():
        for subregion_name, countries in subregions.items():
            for country in countries:
                lookup[country] = {
                    "region": region_name,
                    "subregion" : subregion_name
                }
    return lookup 

print(f'✅ {get_all_countries} defined')
print(f'✅ {build_country_lookup} defined')
print(f'✅ total countries: {len(get_all_countries(FAO_LSIB_REGION))}')


✅ <function get_all_countries at 0x7b60b323c2c0> defined
✅ <function build_country_lookup at 0x7b60b323c400> defined
✅ total countries: 195


In [ ]:
# ── CELL 7: GEE Functions ─────────────────────────────────────────────────────
def prepare_forest_collection(selected_regions, threshold):
    """
    Prepare a GEE Featurecollection with forest area per country for a given canopy threshold
    return GEE object (not yet computed)
    """
    all_countries = get_all_countries(selected_regions)

    lsib_foa=ee.FeatureCollection('USDOS/LSIB_SIMPLE/2017').filter(ee.Filter.inList('country_na', all_countries))
    forest_mask = treecover2000_masked.gte(threshold).selfMask().updateMask(datamask.eq(1))
    area_image = forest_mask.multiply(ee.Image.pixelArea().divide(1e10))

    regions_areas = area_image.reduceRegions(
    collection = lsib_foa,
    reducer = ee.Reducer.sum(),
    scale = 30,
    )   

    regions_areas=regions_areas.map(lambda f: f.set('threshold', threshold))

    return regions_areas

def export_forest_area(selected_regions, thresholds):
    """
    Submit one GEE export task per threshold to Google drive.
    Return list of tasks for monitoring.
    """
    tasks = []
    for threshold in thresholds:
        fc = prepare_forest_collection(selected_regions, threshold)
        task = ee.batch.Export.table.toDrive(
            collection = fc,
            description = f'forest_area_{threshold}',
            folder = 'GEE exports',
            fileNamePrefix = f'forest_area_{threshold}',
            fileFormat='CSV',
            selectors = ['country_na', 'threshold', 'sum']
        )
        task.start()
        tasks.append(task)
        print(f'task submitted for threshold {threshold}')
    
    return tasks

print('✅ prepare_forest_collection() defined')
print('✅ export_forest_area() defined')

✅ prepare_forest_collection() defined
✅ export_forest_area() defined


In [33]:
# ── CELL 8: Run Export ────────────────────────────────────────────────────────
tested_thresholds = [10, 30]
tasks = export_forest_area(FAO_LSIB_REGION,tested_thresholds)


task submitted for threshold 10
task submitted for threshold 30


In [34]:
#  ── CELL 9: Monitor Tasks ────────────────────────────────────────────────────
for task in tasks:
    status = task.status()
    print(f"{status['description']}: {status['state']}")

forest_area_10: COMPLETED
forest_area_30: READY


In [76]:
##  ── CELL 10: forest_area for united states ────────────────────────────────────────────────────

us_state_names = ["Alabama", "Alaska", "Arizona", "Arkansas", "California", "Colorado", "Connecticut", "Delaware",
               "Florida", "Georgia", "Hawaii", "Idaho", "Illinois", "Indiana", "Iowa", "Kansas", "Kentucky",
               "Louisiana", "Maine", "Maryland", "Massachusetts", "Michigan", "Minnesota", "Mississippi", "Missouri",
               "Montana", "Nebraska", "Nevada", "New Hampshire", "New Jersey", "New Mexico", "New York",
               "North Carolina", "North Dakota", "Ohio", "Oklahoma", "Oregon", "Pennsylvania", "Rhode Island",
               "South Carolina", "South Dakota", "Tennessee", "Texas", "Utah", "Vermont", "Virginia", "Washington",
               "West Virginia", "Wisconsin", "Wyoming"]



tiger_state_names =  ee.FeatureCollection('TIGER/2018/States').filter(ee.Filter.inList('NAME', us_state_names))
def prepare_states_forest_collection(selected_states, threshold):
    """same as we did to the countries but here for the USA states"""
    forest_mask_states = treecover2000_masked.gte(threshold).selfMask().updateMask(datamask.eq(1))
    area_image_states = forest_mask_states.multiply(ee.Image.pixelArea().divide(1e10))

    states_areas = area_image_states.reduceRegions(
    collection = tiger_state_names,
    reducer = ee.Reducer.sum(),
    scale = 10,
    maxPixelsPerRegion=1e20   # ← allow more pixels
    )   

    states_areas=states_areas.map(lambda f: f.set('threshold', threshold))

    return states_areas

def export_state_forest_area(selected_states, thresholds):
    """
    Submit one GEE export task per threshold to Google drive.
    Return list of tasks for monitoring.
    """
    tasks = []
    for threshold in thresholds:
        fc = prepare_states_forest_collection(selected_states, threshold)
        task = ee.batch.Export.table.toDrive(
            collection = fc,
            description = f'states_forest_area_{threshold}',
            folder = 'GEE exports',
            fileNamePrefix = f'states_forest_area_{threshold}',
            fileFormat='CSV',
            selectors = ['NAME', 'threshold', 'sum']
        )
        task.start()
        tasks.append(task)
        print(f'task submitted for threshold {threshold}')
    
    return tasks

print('✅ export_state_forest_area() defined')
print('✅ prepare_states_forest_collection() defined')

tested_thresholds = [10, 20, 30, 40, 50]
tasks = export_state_forest_area(tiger_state_names,tested_thresholds)


✅ export_state_forest_area() defined
✅ prepare_states_forest_collection() defined
task submitted for threshold 10
task submitted for threshold 20
task submitted for threshold 30
task submitted for threshold 40
task submitted for threshold 50


In [86]:
#  ── CELL 9: Monitor Tasks ────────────────────────────────────────────────────
for task in tasks:
    status = task.status()
    print(f"{status['description']}: {status['state']}")

df_states_10 = pd.read_csv('/content/drive/MyDrive/GEE exports/states_forest_area_10.csv')
df_states_20 = pd.read_csv('/content/drive/MyDrive/GEE exports/states_forest_area_20.csv')
df_states_30 = pd.read_csv('/content/drive/MyDrive/GEE exports/states_forest_area_30.csv')
df_states_40 = pd.read_csv('/content/drive/MyDrive/GEE exports/states_forest_area_40.csv')
df_states_50 = pd.read_csv('/content/drive/MyDrive/GEE exports/states_forest_area_50.csv')


print(df_states_10.head())

df_states = pd.concat([df_states_10,df_states_20, df_states_30, df_states_40, df_states_50], ignore_index=True)
df_states = df_states.groupby(['NAME', 'threshold'])['sum'].sum().reset_index()
df_states.rename(columns={'NAME': 'state', 'sum': 'area_Mha'}, inplace=True)
df_states.to_csv('/content/drive/MyDrive/GEE exports/states_forest_area_final.csv', index=False)

print(df_states.head())



states_forest_area_10: COMPLETED
states_forest_area_20: COMPLETED
states_forest_area_30: COMPLETED
states_forest_area_40: COMPLETED
states_forest_area_50: COMPLETED
            NAME  threshold       sum
0   Rhode Island         10  0.210974
1  New Hampshire         10  2.144201
2        Vermont         10  2.043624
3    Connecticut         10  1.069887
4          Maine         10  7.397567
     state  threshold  area_Mha
0  Alabama         10  9.617001
1  Alabama         20  9.385570
2  Alabama         30  9.126044
3  Alabama         40  8.899332
4  Alabama         50  8.682608
